# Bengali NLP Safety — V5: train new candidates, keep V2 as the reference

**Best measured Kaggle score: V2 = 0.516. V3 = 0.446; V4 = 0.464. V5 has not been scored.**

## Your next action
1. Upload this notebook to Colab and enable **Runtime → Change runtime type → T4 GPU**.
2. Run Settings, then Installation. If asked to restart, restart, rerun Settings, and continue after Installation.
3. Run the remaining cells in order. Allow Drive access to preserve completed runs.
4. The **Improve datasets and V2 predictions are embedded**. No Kaggle key, CSV upload, or ZIP upload is needed.
5. Two new transformer models train sequentially for six epochs each. Expect potentially several hours, depending on Colab hardware.
6. At the end, read the decision. If a new candidate is selected, submit **submission_v5.csv**. If the notebook retains V2, do not spend another submission on the same predictions.
7. Send **v5_results.zip** and any new Kaggle score. Keep your 0.516 submission as the best until a new one actually beats it.

## What inspecting the supplied data established
- `Improve_train.csv`: **3,297 rows**; validation: **500**; test: **1,000**.
- Improve validation IDs and all 500 label vectors match those recorded in the V4 bundle; V2's saved predictions reproduce validation macro F1 **0.5248511855** on these labels. This is the matching dataset for the saved baseline, unlike the unprefixed/project-source split.
- There are **717 extra repeated-text training rows**, including **385 text groups with conflicting labels**, after conservative whitespace/Unicode normalization. Validation has 102 extra repeated-text rows and 63 conflicting groups.
- There are no exact normalized text overlaps between train, validation, and test in these Improve files. We retain all supplied training rows and labels; we do not resolve ambiguous labels by guessing or discard repeated text+label rows as V3 did.
- **647 training texts and 92 validation texts have no Bengali-script characters**. XLM-R is included as a multilingual diversity experiment; this does not establish it will be better on Romanized Bengali.
- The supplied Improve sample submission still has a different ID set from Improve_test. Predictions use **actual Improve_test IDs**, consistent with your accepted V2–V4 submissions. The mismatch is reported, not silently ignored.
- Two newly trained CPU TF-IDF baselines scored only **0.4817 / 0.4791 fitted validation F1**, below V2. Those results do not justify replacing V2 with TF-IDF.

## Training changes and limits
The first run restores V2-style text handling and length 128 for BanglaBERT while correctly seeding PyTorch. The second trains XLM-R base using conservative raw text. Both use all 3,297 training rows, unweighted multilabel BCE, dynamic padding, mixed precision, and sample-correct gradient accumulation.

Six epochs are predeclared because both V3 runs were still reaching their best recorded development score at epoch 5. Snapshots at epochs 4, 5, and 6 are compared, together with an average of epochs 5–6. Validation labels do not enter gradient training, and no validation-based early stopping changes the schedule.

This is a new experiment, not a guaranteed improvement. V2's model weights are unavailable; it is retained through its real saved probabilities, not an assumed exact retraining reproduction.

**Rules assumption:** seven binary labels and macro F1 are inherited from the original notebook; confirm the competition permits these pretrained models. No extra labeled corpus is added and no labels are copied from the other dataset split.

In [ ]:
# 1. Settings — use the defaults for the first run.
from pathlib import Path
SEED = 42
TRAIN_NEW_MODELS = True
USE_DRIVE = True
EPOCHS = 6
SNAPSHOT_EPOCHS = [4, 5, 6]
TRAIN_BATCH_SIZE = 8
ACCUMULATION_STEPS = 2
EVAL_BATCH_SIZE = 16
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
REUSE_COMPLETED_RUNS = True
SAVE_MODEL_WEIGHTS = False            # True saves each snapshot, requiring several GB of Drive space
BOOTSTRAPS = 8
N_FOLDS = 5
MIN_GAIN = 0.01                       # minimum pooled development macro-F1 gain over unchanged V2
MIN_WINNING_FOLDS = 3
PIPELINE_VERSION = "v5.0_improve"
MODELS = [
    {"name":"banglabert_v2style", "model":"csebuetnlp/banglabert", "view":"v2_text", "max_length":128, "lr":2e-5, "seed":42},
    {"name":"xlmr_base", "model":"FacebookAI/xlm-roberta-base", "view":"clean_text", "max_length":128, "lr":2e-5, "seed":42},
]
OUTPUT_ROOT = Path("/content/drive/MyDrive/bengali_safety_v5" if USE_DRIVE else "/content/bengali_safety_v5")
LABELS = ["is_toxic","implicit_toxicity","sarcasm","contains_profanity","threatening","identity_attack","sexual_content"]
ID_COL, TEXT_COL = "ID", "text"
assert EPOCHS >= max(SNAPSHOT_EPOCHS) and {4,5,6}.issubset(SNAPSHOT_EPOCHS)
assert TRAIN_BATCH_SIZE > 0 and ACCUMULATION_STEPS > 0
print("Planned new fits:",len(MODELS) if TRAIN_NEW_MODELS else 0)

In [ ]:
# 2. Installation — keep Colab's GPU-enabled PyTorch.
import sys,subprocess
subprocess.check_call([sys.executable,"-m","pip","install",
    "transformers==4.57.6","tokenizers==0.22.1","sentencepiece==0.2.1",
    "iterative-stratification==0.1.9","scikit-learn","pandas","tqdm","safetensors"])
print("If prompted to restart, restart and rerun Settings, then continue below.")

In [ ]:
# 3. Exact supplied Improve files and V2 baseline; no API credentials.
import base64,io,zipfile,hashlib,json
BUNDLE_SHA256 = '6156f1fd89e2be6b06e66922d93480be3bd7ad6bcbf5ceb66046586022ef9188'
MANIFEST = {'train.csv': 'eb0a9da9847079ed840a1994150a7fbf41bbba023ad21e893f7b4e8c601808c4', 'validation.csv': '71e177080ef240f9d51317f291bbb0a21168ecc2042041dd33f3f3eabb90c8e3', 'test.csv': 'a8e2d780dca9a79ad7220d8f0fb29bf74f11a7060454bb2bbd685be0cac39a1f', 'sample_submission.csv': '382cd535b99becc6eb68e837f84f640a94030e9c636da471f439519b7ba97cc0', 'v2_config.json': '6581c6afac441173384023b252d4ffcbb68bc6f304d3a8cb53c6cf328c9ba7f1', 'v2_valid.csv': '52d7d4f0555f501e14f51f30a013ea1df23277c3951cbb6cd907e5d093567a52', 'v2_test.csv': '12fe7e461faab5caab7778db73d45c00f7249623c25e4ce67a88a222e10f26e1', 'linear_predictions.npz': '83f6e9733c24ab5854ac460a75b47aa229b1c7231d26f5c72e3c431797c04687'}
BUNDLE_BASE64 = '...'
BUNDLE = base64.b64decode(BUNDLE_BASE64)
assert hashlib.sha256(BUNDLE).hexdigest()==BUNDLE_SHA256
print("Bundled inputs verified.")

In [ ]:
# 4. Read, verify and align data. No model or prediction from the other split is mixed in.
import gc,re,random,math,time,unicodedata,importlib.metadata,warnings
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset,DataLoader
from transformers import AutoTokenizer,AutoModelForSequenceClassification,get_linear_schedule_with_warmup
from sklearn.metrics import f1_score,precision_recall_curve,precision_recall_fscore_support
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from tqdm.auto import tqdm
from IPython.display import display
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(BUNDLE)) as archive:
    def get_bytes(name):
        data=archive.read(name)
        assert hashlib.sha256(data).hexdigest()==MANIFEST[name]
        return data
    def read_csv(name):return pd.read_csv(io.BytesIO(get_bytes(name)),dtype={ID_COL:str})
    train_df=read_csv("train.csv");valid_df=read_csv("validation.csv");test_df=read_csv("test.csv");sample=read_csv("sample_submission.csv")
    v2_valid_frame=read_csv("v2_valid.csv");v2_test_frame=read_csv("v2_test.csv")
    v2_config=json.loads(get_bytes("v2_config.json"))
    with np.load(io.BytesIO(get_bytes("linear_predictions.npz")),allow_pickle=False) as z:
        linear_predictions={k:z[k].copy() for k in z.files}
for frame,labeled in [(train_df,True),(valid_df,True),(test_df,False)]:
    assert frame[ID_COL].notna().all() and frame[ID_COL].is_unique
    assert frame[TEXT_COL].notna().all()
    if labeled:assert frame[LABELS].isin([0,1]).all().all()
assert set(train_df[ID_COL]).isdisjoint(valid_df[ID_COL])
assert set(train_df[ID_COL]).isdisjoint(test_df[ID_COL])
assert set(valid_df[ID_COL]).isdisjoint(test_df[ID_COL])
assert set(sample.columns)=={ID_COL,*LABELS} and sample[ID_COL].is_unique
sample_matches=set(sample[ID_COL])==set(test_df[ID_COL]) and len(sample)==len(test_df)
submission_ids=sample[ID_COL].tolist() if sample_matches else test_df[ID_COL].tolist()
if not sample_matches:print("Known sample-ID mismatch: using actual Improve_test IDs, as in accepted submissions.")

def clean_text(value):
    return re.sub(r"\s+"," ",unicodedata.normalize("NFC",str(value))).strip()
for frame in [train_df,valid_df,test_df]:
    frame["clean_text"]=frame[TEXT_COL].map(clean_text)
    frame["v2_text"]=frame[TEXT_COL].map(lambda value:re.sub(r"\s+"," ",str(value).replace("\u200c"," ").replace("\u200d"," ")).strip())
    frame["group"]=frame.clean_text
assert set(train_df.group).isdisjoint(valid_df.group)

def align_probabilities(frame,ids):
    assert frame[ID_COL].is_unique and set(frame[ID_COL])==set(ids)
    result=frame.set_index(ID_COL).loc[ids,[l+"_prob" for l in LABELS]].to_numpy(dtype=np.float64)
    assert np.isfinite(result).all() and ((result>=0)&(result<=1)).all()
    return result
v2_valid=align_probabilities(v2_valid_frame,valid_df[ID_COL].tolist())
v2_test=align_probabilities(v2_test_frame,test_df[ID_COL].tolist())
v2_thresholds=np.array([v2_config["thresholds"][l] for l in LABELS])
y_train=train_df[LABELS].to_numpy(dtype=np.float32)
y_valid=valid_df[LABELS].to_numpy(dtype=np.int8)
v2_score=float(f1_score(y_valid,v2_valid>=v2_thresholds,average="macro",zero_division=0))
assert np.isclose(v2_score,v2_config["validation_macro_f1"],atol=1e-9)
print("V2 reproduced validation macro F1:",v2_score,"| reported Kaggle: 0.516")
print("Training/validation/test rows:",len(train_df),len(valid_df),len(test_df))

In [ ]:
# 5. Preserve all supplied labels; report conflicts rather than invent corrections.
conflict_counts={}
for name,frame in [("train",train_df),("validation",valid_df)]:
    group_counts=frame.groupby("group")[LABELS].nunique()
    conflicts=group_counts.gt(1).any(axis=1)
    conflict_counts[name]={"extra_repeated_text_rows":int(frame.group.duplicated().sum()),"conflicting_text_groups":int(conflicts.sum())}
    print(name,conflict_counts[name])
display(pd.DataFrame({"train_positives":y_train.sum(0),"validation_positives":y_valid.sum(0)},index=LABELS))

def grouped_folds(frame,n_splits=5,seed=42):
    grouped=frame.groupby("group",sort=True)[LABELS].max()
    assert len(grouped)>=n_splits
    splitter=MultilabelStratifiedKFold(n_splits=n_splits,shuffle=True,random_state=seed)
    mapping={}
    for fold,(_,held) in enumerate(splitter.split(np.zeros((len(grouped),1)),grouped.values)):
        for key in grouped.index[held]:mapping[key]=fold
    result=frame.group.map(mapping).to_numpy(dtype=int)
    for fold in range(n_splits):
        assert set(frame.loc[result==fold,"group"]).isdisjoint(frame.loc[result!=fold,"group"])
    return result
fold_ids=grouped_folds(valid_df,N_FOLDS,SEED)
versions={p:importlib.metadata.version(p) for p in ["torch","transformers","tokenizers","numpy","scikit-learn","iterative-stratification"]}
settings={"pipeline":PIPELINE_VERSION,"bundle":BUNDLE_SHA256,"epochs":EPOCHS,"snapshots":SNAPSHOT_EPOCHS,
    "batch":TRAIN_BATCH_SIZE,"accumulation":ACCUMULATION_STEPS,"warmup":WARMUP_RATIO,"weight_decay":WEIGHT_DECAY,
    "save_weights":SAVE_MODEL_WEIGHTS,"versions":versions}
run_key=hashlib.sha256(json.dumps(settings,sort_keys=True).encode()).hexdigest()[:16]
RUN_DIR=OUTPUT_ROOT/run_key;RUN_DIR.mkdir(parents=True,exist_ok=True)
valid_df[[ID_COL]].assign(fold=fold_ids).to_csv(RUN_DIR/"validation_folds.csv",index=False)
(RUN_DIR/"dataset_diagnostics.json").write_text(json.dumps({"manifest":MANIFEST,"conflicts":conflict_counts,"sample_matches":sample_matches},indent=2))
print("Output folder:",RUN_DIR)

## Why one run uses V2-style normalization
The BanglaBERT authors recommend their own normalization pipeline. V3 already added it, alongside several other changes, and its trained models were weaker than V2. That does not prove normalization caused the regression. This experiment deliberately restores V2's input handling for one BanglaBERT run to reduce changes from the successful baseline. A controlled normalization ablation can follow after seeing these results.

XLM-R uses conservative Unicode/whitespace cleaning. It is an additional trained model, not an assumption that multilingual automatically means better. No auxiliary language model generates labels or rewrites the dataset.

In [ ]:
# 6. Correct positive-label thresholds and repeated-text-aware comparisons.
def positive_f1(target,prediction):
    return float(f1_score(np.asarray(target).reshape(-1),np.asarray(prediction).reshape(-1),average="binary",zero_division=0))

def best_threshold(target,probability):
    if target.sum()==0:return 1.000001
    if target.sum()==len(target):return 0.0
    p,r,t=precision_recall_curve(target,probability)
    score=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)
    return float(t[np.flatnonzero(np.isclose(score,score.max(),atol=1e-12,rtol=0))[-1]])

def fit_thresholds(target,probability,bootstraps=BOOTSTRAPS):
    assert target.shape==probability.shape and np.isfinite(probability).all()
    rng=np.random.default_rng(SEED)
    estimates=[[best_threshold(target[:,j],probability[:,j]) for j in range(len(LABELS))]]
    for _ in range(bootstraps):
        idx=rng.integers(0,len(target),len(target))
        estimates.append([best_threshold(target[idx,j],probability[idx,j]) for j in range(len(LABELS))])
    return np.median(estimates,axis=0)

def macro(target,prediction):return float(f1_score(target,prediction,average="macro",zero_division=0))

def compare_candidate(probability):
    prediction=np.zeros_like(y_valid)
    fold_scores=[]
    for fold in range(N_FOLDS):
        held=fold_ids==fold
        t=fit_thresholds(y_valid[~held],probability[~held])
        prediction[held]=(probability[held]>=t).astype(np.int8)
        fold_scores.append(macro(y_valid[held],prediction[held]))
    return macro(y_valid,prediction),prediction,fold_scores

print("Historical V2 baseline:",v2_score)
for name in ["linear_C1","linear_C4"]:
    prob=linear_predictions[name+"_valid"]
    print(name,"cross-fitted threshold diagnostic:",compare_candidate(prob)[0])

In [ ]:
# 7. Dynamic-padding transformer datasets and inference.
class EncodedDataset(Dataset):
    def __init__(self,encoded,targets=None):self.encoded=encoded;self.targets=targets
    def __len__(self):return len(self.encoded["input_ids"])
    def __getitem__(self,index):
        item={key:value[index] for key,value in self.encoded.items()}
        if self.targets is not None:item["labels"]=self.targets[index]
        return item
class LabelCollator:
    def __init__(self,tokenizer):self.tokenizer=tokenizer
    def __call__(self,items):
        batch=self.tokenizer.pad([{k:v for k,v in item.items() if k!="labels"} for item in items],padding=True,pad_to_multiple_of=8,return_tensors="pt")
        if "labels" in items[0]:batch["labels"]=torch.tensor(np.stack([item["labels"] for item in items]),dtype=torch.float32)
        return batch

def make_loader(frame,tokenizer,cfg,training=False):
    encoded=tokenizer(frame[cfg["view"]].tolist(),truncation=True,max_length=cfg["max_length"],padding=False)
    dataset=EncodedDataset(encoded,y_train if training else None)
    return DataLoader(dataset,batch_size=TRAIN_BATCH_SIZE if training else EVAL_BATCH_SIZE,shuffle=training,
        num_workers=0,pin_memory=True,collate_fn=LabelCollator(tokenizer))

def seed_everything(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False

@torch.no_grad()
def predict(model,loader):
    model.eval();parts=[]
    for batch in loader:
        inputs={k:v.to("cuda") for k,v in batch.items() if k!="labels"}
        with torch.autocast(device_type="cuda",dtype=torch.float16):logits=model(**inputs).logits
        parts.append(torch.sigmoid(logits.float()).cpu().numpy())
    out=np.concatenate(parts)
    assert np.isfinite(out).all() and ((out>=0)&(out<=1)).all()
    return out

In [ ]:
# 8. Six fixed epochs; save paired validation/test snapshots. No model remains on GPU between runs.
def train_one(cfg):
    key=hashlib.sha256(json.dumps(cfg,sort_keys=True).encode()).hexdigest()[:10]
    folder=RUN_DIR/(cfg["name"]+"_"+key);folder.mkdir(exist_ok=True)
    cache,meta_path=folder/"predictions.npz",folder/"run.json"
    if REUSE_COMPLETED_RUNS and cache.exists() and meta_path.exists():
        meta=json.loads(meta_path.read_text());assert meta["config"]==cfg and meta["settings"]==settings
        with np.load(cache,allow_pickle=False) as z:vp=z["valid"].copy();tp=z["test"].copy()
        assert vp.shape==(len(SNAPSHOT_EPOCHS),len(valid_df),len(LABELS)) and tp.shape==(len(SNAPSHOT_EPOCHS),len(test_df),len(LABELS))
        assert np.isfinite(vp).all() and np.isfinite(tp).all()
        print("Reusing completed run:",cfg["name"])
        return {"config":cfg,"valid":vp,"test":tp,"meta":meta}
    assert torch.cuda.is_available(), "Enable T4 GPU in Colab before training."
    seed_everything(cfg["seed"])
    tokenizer=AutoTokenizer.from_pretrained(cfg["model"])
    probe=tokenizer(train_df[cfg["view"]].sample(min(1000,len(train_df)),random_state=SEED).tolist(),truncation=False)
    lengths=np.array([len(x) for x in probe["input_ids"]])
    print(cfg["name"],"token length p50/p95/p99",np.percentile(lengths,[50,95,99]),"truncated fraction",float((lengths>cfg["max_length"]).mean()))
    train_loader=make_loader(train_df,tokenizer,cfg,True)
    valid_loader=make_loader(valid_df,tokenizer,cfg)
    test_loader=make_loader(test_df,tokenizer,cfg)
    model=AutoModelForSequenceClassification.from_pretrained(cfg["model"],num_labels=len(LABELS),
        problem_type="multi_label_classification",id2label=dict(enumerate(LABELS)),label2id={l:i for i,l in enumerate(LABELS)}).to("cuda")
    groups=[{"params":[p for p in model.parameters() if p.requires_grad and p.ndim>1],"weight_decay":WEIGHT_DECAY},
            {"params":[p for p in model.parameters() if p.requires_grad and p.ndim<=1],"weight_decay":0.0}]
    optimizer=torch.optim.AdamW(groups,lr=cfg["lr"])
    steps=math.ceil(len(train_loader)/ACCUMULATION_STEPS)*EPOCHS
    scheduler=get_linear_schedule_with_warmup(optimizer,int(steps*WARMUP_RATIO),steps)
    scaler=torch.amp.GradScaler("cuda")
    loss_fn=torch.nn.BCEWithLogitsLoss()
    snapshots={};history=[];start=time.time()
    for epoch in range(1,EPOCHS+1):
        model.train();optimizer.zero_grad(set_to_none=True);loss_total=0.
        for step,batch in enumerate(tqdm(train_loader,desc=f"{cfg['name']} epoch {epoch}/{EPOCHS}")):
            target=batch.pop("labels").to("cuda")
            inputs={k:v.to("cuda") for k,v in batch.items()}
            group_start=(step//ACCUMULATION_STEPS)*ACCUMULATION_STEPS
            sample_count=min(TRAIN_BATCH_SIZE*ACCUMULATION_STEPS,len(train_df)-group_start*TRAIN_BATCH_SIZE)
            with torch.autocast(device_type="cuda",dtype=torch.float16):logits=model(**inputs).logits
            loss=loss_fn(logits.float(),target)
            if not torch.isfinite(loss):raise FloatingPointError("Training loss is not finite.")
            scaler.scale(loss*(len(target)/sample_count)).backward()
            loss_total+=float(loss.item())*len(target)
            if (step+1)%ACCUMULATION_STEPS==0 or step+1==len(train_loader):
                scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()
                if scaler.get_scale()>=old_scale:scheduler.step()
                optimizer.zero_grad(set_to_none=True)
        row={"epoch":epoch,"loss":loss_total/len(train_df),"elapsed_seconds":time.time()-start}
        if epoch in SNAPSHOT_EPOCHS:
            vp=predict(model,valid_loader);tp=predict(model,test_loader);snapshots[epoch]=(vp,tp)
            row["validation_f1_at_0_5"]=macro(y_valid,vp>=.5)  # logged only, not an early-stopping rule
            if SAVE_MODEL_WEIGHTS:
                model.save_pretrained(folder/f"epoch_{epoch}",safe_serialization=True);tokenizer.save_pretrained(folder/f"epoch_{epoch}")
        history.append(row);pd.DataFrame(history).to_csv(folder/"history.csv",index=False);print(row)
    vp=np.stack([snapshots[e][0] for e in SNAPSHOT_EPOCHS]);tp=np.stack([snapshots[e][1] for e in SNAPSHOT_EPOCHS])
    meta={"config":cfg,"settings":settings,"history":history,"resolved_model_commit":getattr(model.config,"_commit_hash",None)}
    temp=folder/"predictions.tmp.npz";np.savez_compressed(temp,valid=vp,test=tp);temp.replace(cache)
    meta_path.write_text(json.dumps(meta,indent=2))
    del model,optimizer,scheduler,scaler,groups,train_loader,valid_loader,test_loader,tokenizer
    gc.collect();torch.cuda.empty_cache()
    return {"config":cfg,"valid":vp,"test":tp,"meta":meta}

results=[]
if TRAIN_NEW_MODELS:
    for cfg in MODELS:results.append(train_one(cfg))
else:
    print("New training disabled. The output will retain V2; this is not a new model improvement.")

## How a new model can replace V2
For each candidate, its thresholds are fitted on four validation folds and evaluated on the fifth, keeping repeated text groups together. The comparison includes individual epoch snapshots, a two-epoch average, a mean across the new model families, and conservative **80% V2 + 20% new-candidate** blends.

The default gate requires at least **0.01 pooled macro-F1 improvement** over unchanged V2 and wins in at least **three of five folds**. This is a cautious experiment filter, not statistical proof or a guarantee of improvement. If no candidate qualifies, V2 is retained.

**Evaluation limitation:** V2 was already selected and threshold-tuned on this validation set, and several earlier experiments also used it. Comparing multiple candidates introduces further selection optimism. These are development diagnostics, not an independent test or a promised Kaggle score. Test labels are never used. Do not repeatedly change the gate to force a new submission.

In [ ]:
# 9. Matching probability ensembles; model/snapshot identity is preserved at test time.
def build_candidates(results):
    items=[];means=[]
    for result in results:
        name=result["config"]["name"]
        for epoch in SNAPSHOT_EPOCHS:
            i=SNAPSHOT_EPOCHS.index(epoch)
            items.append({"name":name+f"_epoch_{epoch}","valid":result["valid"][i],"test":result["test"][i]})
        idx=[SNAPSHOT_EPOCHS.index(e) for e in [5,6]]
        mean={"name":name+"_mean_5_6","valid":result["valid"][idx].mean(0),"test":result["test"][idx].mean(0)}
        items.append(mean);means.append(mean)
    if len(means)>1:items.append({"name":"new_models_mean_5_6","valid":np.mean([x["valid"] for x in means],axis=0),"test":np.mean([x["test"] for x in means],axis=0)})
    for item in list(items):
        items.append({"name":"v2_80pct_plus_"+item["name"],"valid":.8*v2_valid+.2*item["valid"],"test":.8*v2_test+.2*item["test"]})
    return items
candidates=build_candidates(results)
v2_pred=v2_valid>=v2_thresholds
baseline_fold_scores=np.array([macro(y_valid[fold_ids==f],v2_pred[fold_ids==f]) for f in range(N_FOLDS)])
selected={"name":"unchanged_v2","valid":v2_valid,"test":v2_test,"thresholds":v2_thresholds,"score":v2_score,"diagnostic_prediction":v2_pred}
rows=[{"candidate":"unchanged_v2","development_macro_f1":v2_score,"winning_folds":0,"passes_gate":False}]
for candidate in tqdm(candidates,desc="Compare candidates against V2"):
    score,prediction,scores=compare_candidate(candidate["valid"])
    wins=int((np.array(scores)>baseline_fold_scores+1e-10).sum())
    passes=score>=v2_score+MIN_GAIN and wins>=MIN_WINNING_FOLDS
    rows.append({"candidate":candidate["name"],"development_macro_f1":score,"winning_folds":wins,"passes_gate":passes})
    if passes and score>selected["score"]:
        selected={**candidate,"thresholds":fit_thresholds(y_valid,candidate["valid"]),"score":score,"diagnostic_prediction":prediction}
comparison=pd.DataFrame(rows).sort_values("development_macro_f1",ascending=False)
comparison.to_csv(RUN_DIR/"candidate_comparison.csv",index=False)
display(comparison)
print("SELECTED:",selected["name"])
if selected["name"]=="unchanged_v2":print("No tested candidate passed the gate. Output restores V2; do not claim or resubmit it as an improvement.")
else:print("New candidate passed the local gate. Kaggle performance is still unmeasured.")

In [ ]:
# 10. Per-label diagnostics, valid CSV, and complete results for the next review.
p,r,f,s=precision_recall_fscore_support(y_valid,selected["diagnostic_prediction"],zero_division=0)
per_label=pd.DataFrame({"label":LABELS,"positives":s,"precision":p,"recall":r,"f1":f,"final_threshold":selected["thresholds"],
    "test_predicted_positives":(selected["test"]>=selected["thresholds"]).sum(0)})
per_label.to_csv(RUN_DIR/"per_label_report.csv",index=False);display(per_label.round(4))

def make_submission(probability,thresholds,ids,ordered_ids):
    assert probability.shape==(len(ids),len(LABELS)) and np.isfinite(probability).all()
    assert ((probability>=0)&(probability<=1)).all()
    assert len(set(ids))==len(ids) and len(set(ordered_ids))==len(ordered_ids) and set(ids)==set(ordered_ids)
    prediction=(probability>=thresholds).astype(np.int8)
    frame=pd.DataFrame(prediction,index=pd.Index(ids,name=ID_COL),columns=LABELS)
    frame=frame.loc[ordered_ids].reset_index()[sample.columns.tolist()]
    assert frame[ID_COL].tolist()==ordered_ids and frame[LABELS].isin([0,1]).all().all()
    return frame
submission=make_submission(selected["test"],selected["thresholds"],test_df[ID_COL].tolist(),submission_ids)
SUBMISSION_PATH=RUN_DIR/"submission_v5.csv";submission.to_csv(SUBMISSION_PATH,index=False)
roundtrip=pd.read_csv(SUBMISSION_PATH,dtype={ID_COL:str})
assert roundtrip[ID_COL].tolist()==submission_ids and np.array_equal(roundtrip[LABELS],submission[LABELS])
report={"version":PIPELINE_VERSION,"manifest":MANIFEST,"settings":settings,"models":MODELS,
    "prior_kaggle":{"v2":.516,"v3":.446,"v4":.464},"new_kaggle_score":None,
    "selected":selected["name"],"is_new_candidate":selected["name"]!="unchanged_v2",
    "development_macro_f1":selected["score"],"original_v2_validation_f1":v2_score,
    "thresholds":dict(zip(LABELS,selected["thresholds"].tolist())),"minimum_gain":MIN_GAIN,"minimum_winning_folds":MIN_WINNING_FOLDS,
    "sample_matches":sample_matches,"conflicts":conflict_counts,"runs":[r["meta"] for r in results],
    "limitation":"Reused validation set; development comparison is not an independent Kaggle estimate."}
(RUN_DIR/"experiment_report.json").write_text(json.dumps(report,indent=2))
np.savez_compressed(RUN_DIR/"selected_predictions.npz",valid=selected["valid"],test=selected["test"],labels=y_valid,label_names=np.array(LABELS),
    valid_ids=valid_df[ID_COL].to_numpy(dtype=str),test_ids=test_df[ID_COL].to_numpy(dtype=str),thresholds=selected["thresholds"])
BUNDLE_PATH=RUN_DIR/"v5_results.zip"
with zipfile.ZipFile(BUNDLE_PATH,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for path in RUN_DIR.iterdir():
        if path.is_file() and path.suffix in {".csv",".json",".npz"}:z.write(path,path.name)
    for path in RUN_DIR.glob("*/predictions.npz"):z.write(path,str(path.relative_to(RUN_DIR)))
    for path in RUN_DIR.glob("*/run.json"):z.write(path,str(path.relative_to(RUN_DIR)))
print("Saved:",SUBMISSION_PATH,BUNDLE_PATH,sep="\n")
from google.colab import files
files.download(str(SUBMISSION_PATH));files.download(str(BUNDLE_PATH))

## Troubleshooting
- **GPU memory error:** restart, set training batch to 4, accumulation to 4, evaluation batch to 8, and rerun. Different training settings create a new cache directory.
- **Disconnect:** rerun after mounting Drive. Completed runs are reused; an interrupted run restarts from epoch 1. No optimizer-state resume is implemented.
- **No new candidate selected:** the output is V2, not a new improvement. Send the results ZIP; don't repeatedly relax the gate.
- **Dataset change:** this notebook intentionally uses the embedded Improve files. A different dataset requires a separately checked revision; it will not silently download different CSVs.

## Verification and sources
The Improve CSVs and saved V2 baseline were inspected locally. V2's validation score was reproduced, and two full-training-set TF-IDF candidates were actually fitted and evaluated on these files. Notebook syntax, input hashes/alignment, thresholds, group-to-fold mapping with a fixture splitter, candidate/test averaging, fallback behavior, and submission schema were checked. The external stratification package itself could not be installed in this local environment and will run in Colab. GPU training of the new transformer candidates has **not** run here; no V5 Kaggle score is claimed.

- [BanglaBERT model card](https://huggingface.co/csebuetnlp/banglabert/blob/main/README.md): pretrained model and recommended normalization; the V2-style ablation is explicitly documented above.
- [XLM-R base model card](https://huggingface.co/FacebookAI/xlm-roberta-base/blob/main/README.md): multilingual pretrained model used as the diversity candidate.
- [scikit-learn F1](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html): use positive-class binary F1 for one label, macro F1 across all seven.